In [1]:
import traitlets
import threading
import numpy as np
import cv2
import atexit

class Camera(traitlets.HasTraits):
    value = traitlets.Any()
    width = traitlets.Integer(default_value=224)
    height = traitlets.Integer(default_value=224)
    format = traitlets.Unicode(default_value='bgr8')
    running = traitlets.Bool(default_value=False)
    
    def __init__(self, *args, **kwargs):
        super(Camera, self).__init__(*args, **kwargs)
        if self.format == 'bgr8':
            self.value = np.empty((self.height, self.width, 3), dtype=np.uint8)
        self._running = False
            
    def _read(self):
        raise NotImplementedError
        
    def read(self):
        if self._running:
            raise RuntimeError('Cannot read directly while camera is running')
        self.value = self._read()
        return self.value
    
    def _capture_frames(self):
        while True:
            if not self._running:
                break
            self.value = self._read()
            
    @traitlets.observe('running')
    def _on_running(self, change):
        if change['new'] and not change['old']:
            self._running = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()
        elif change['old'] and not change['new']:
            self._running = False
            self.thread.join()

def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])


class CSICamera(Camera):
    
    capture_fps = traitlets.Integer(default_value=30)
    capture_width = traitlets.Integer(default_value=640)
    capture_height = traitlets.Integer(default_value=480)
    capture_device = traitlets.Integer(default_value=0)
    
    def __init__(self, *args, **kwargs):
        super(CSICamera, self).__init__(*args, **kwargs)
        try:
            self.cap = cv2.VideoCapture(self._gst_str(), cv2.CAP_GSTREAMER)
            
            if not self.cap.isOpened():
                raise RuntimeError('Could not open camera with GStreamer pipeline.')
            
            re, image = self.cap.read()
            if not re:
                raise RuntimeError('Could not read image from camera.')
            
        except Exception as e:
            raise RuntimeError(
                'Could not initialize camera: {}'.format(str(e)))
        
        atexit.register(self.cap.release)
    
    def _gst_str(self):
        return (
            'nvarguscamerasrc sensor-id={} ! '
            'video/x-raw(memory:NVMM), width=(int){}, height=(int){}, '
            'format=(string)NV12, framerate=(fraction){}/1 ! '
            'nvvidconv flip-method=0 ! '
            'video/x-raw, width=(int){}, height=(int){}, format=(string)BGRx ! '
            'videoconvert ! '
            'video/x-raw, format=(string)BGR ! '
            'appsink max-buffers=1 drop=true'
        ).format(
            self.capture_device,
            self.capture_width,
            self.capture_height,
            self.capture_fps,
            self.capture_width,
            self.capture_height
        )
    
    def _read(self):
        re, image = self.cap.read()
        if re:
            image_resized = cv2.resize(image, (int(self.width), int(self.height)))
            return image_resized
        else:
            raise RuntimeError('Could not read image from camera')

In [2]:
camera = CSICamera(width=224, height=224, capture_width=640, capture_height=480, capture_device=0)
image = camera.read()
print(image.shape)

(224, 224, 3)


In [3]:
import ipywidgets
from IPython.display import display
#import bgr8_to_jpeg

image_widget = ipywidgets.Image(format='jpeg')

#this is using the image defined in the previous code block that is reading the camera stream
#reading one image
image_widget.value = bgr8_to_jpeg(image)

display(image_widget)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [4]:
camera.running = True

def update_image(change):
    image = change['new']
    image_widget.value = bgr8_to_jpeg(image)
    
camera.observe(update_image, names='value')

In [5]:
camera.unobserve(update_image, names='value')

In [6]:
#lets transform the image
camera.running = True

def update_image(change):
    image = change['new']
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    image_widget.value = bgr8_to_jpeg(gray)
    
camera.observe(update_image, names='value')

In [7]:
#lets transform the image the CUDA way
camera.running = True

def update_image(change):
    image = change['new']
    gpu_frame = cv2.cuda_GpuMat()
    gpu_frame.upload(image)
    gray = cv2.cuda.cvtColor(gpu_frame, cv2.COLOR_BGR2GRAY)
    gray = gray.download()
    image_widget.value = bgr8_to_jpeg(gray)
    
camera.observe(update_image, names='value')

In [8]:
camera.unobserve(update_image, names='value')

In [10]:
from IPython.display import display
from ipywidgets import Video, Image
import numpy as np
import base64

video = Video.from_file('sample-5s.mp4')
video

Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free\x00+W\xdfmdat\x00\x00\…

In [15]:
import cv2
import numpy as np

cap = cv2.VideoCapture('sample-5s.mp4')
frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)           # Canny expects grayscale input
    
    # Convert single channel edges back to BGR so VideoWriter can save it
    edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    frames.append(edges_bgr)

width = int(cap.get(3))
height = int(cap.get(4))
cap.release()

filename = 'output.mp4'
fourcc = cv2.VideoWriter_fourcc(*'avc1')
writer = cv2.VideoWriter(filename, fourcc, 25, (width, height))

for frame in frames:
    writer.write(frame)

writer.release()

with open(filename, 'rb') as f:
    video.value = f.read()

In [16]:
import pycuda.driver as drv
from pycuda.compiler import SourceModule
import numpy as np
import cv2
import pycuda.autoinit

mod = SourceModule(
    """
#include<stdio.h>
#define INDEX(a, b) a*256+b
__global__ void bgr2gray(float *d_result, float *b_img, float *g_img, float *r_img)
{
    unsigned int idx = threadIdx.x + (blockIdx.x * (blockDim.x * blockDim.y));
    unsigned int a = idx / 256;
    unsigned int b = idx % 256;
    d_result[INDEX(a, b)] = (0.299f * r_img[INDEX(a, b)] +
                              0.587f * g_img[INDEX(a, b)] +
                              0.114f * b_img[INDEX(a, b)]);
}
"""
)

cap = cv2.VideoCapture('sample-5s.mp4')  # <-- replaced gst_str()
bgr2gray = mod.get_function("bgr2gray")

while True:
    ret, h_img = cap.read()
    if not ret:
        print("End of video or failed to grab frame")
        break

    h_img = cv2.resize(h_img, (256, 256), interpolation=cv2.INTER_CUBIC)
    b_img = h_img[:, :, 0].reshape(65536).astype(np.float32)
    g_img = h_img[:, :, 1].reshape(65536).astype(np.float32)
    r_img = h_img[:, :, 2].reshape(65536).astype(np.float32)
    h_result = np.empty_like(r_img)

    bgr2gray(drv.Out(h_result), drv.In(b_img), drv.In(g_img), drv.In(r_img),
             block=(1024, 1, 1), grid=(64, 1, 1))

    h_result = np.reshape(h_result, (256, 256)).astype(np.uint8)
    cv2.imshow("Grayscale Image", h_result)
    cv2.imshow("Original frame", h_img)

    if cv2.waitKey(50) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

error: OpenCV(4.5.0) /opt/opencv/modules/highgui/src/window_gtk.cpp:641: error: (-2:Unspecified error) GTK backend is not available in function 'cvInitSystem'
